# RAG Enhancements
Experimenting with different ways of RAG enhancements
These enhancements include:
- Input Enhancements
- Retriever Enhancements
- Generator Enhancements
- RAG Pipeline Enhancements

## Imports

In [ ]:
import chromadb
from google import genai
import os
import json
from transformers import (
    DPRContextEncoder, DPRContextEncoderTokenizer, 
    DPRQuestionEncoder, DPRQuestionEncoderTokenizer
    )
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import typing
from pydantic import BaseModel, Field
import time

In [3]:
import torch, torchvision
print(torch.cuda.is_available(), torch.version.cuda)
from torchvision.ops import nms
print(nms)

True 12.8
<function nms at 0x0000020772B90360>


### Loading environment variables
Gemini api key, chromadb database client, llm client

In [4]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set in the environment or .env file.")

DB_PATH = "./avatar_rag_db"

def get_db_client():
    """Returns a PersistentClient for ChromaDB."""
    return chromadb.PersistentClient(path=DB_PATH)

def get_llm_client():
    """Returns the GenAI client."""
    return genai.Client(api_key=GEMINI_API_KEY)

### Creating database 

In [5]:
data_path = "../data/true_synthetic_requests.json"
with open(data_path, "r") as f:
    data = json.load(f)

# Extract lists for ChromaDB ingestion
documents = [item["document"] for item in data]
metadatas = [item["metadata"] for item in data]
ids = [item["id"] for item in data]

## Retriever Enhancement
- DPR
- Recursive Retrieval
- Retriever Fine-tuning
- Hybrid Retrieval
- Reranking

### Dense Passage Retrieval (DPR)

In [6]:
ctx_tok = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_enc = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
doc_emb = ctx_enc(**ctx_tok(documents, return_tensors="pt", padding=True, truncation=True)).pooler_output.detach().numpy()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 131384.62it/s]
DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-single-nq-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Created the document embedder using DPR models from huggingface

In [7]:
client = get_db_client()
collection_dpr = client.get_or_create_collection(name="collection_dpr")

# Inject all 100 items
collection_dpr.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=doc_emb # embedding_dim=768 compared to 384 with MiniLM
)

print(f"Successfully injected {len(documents)} requests into the vector database.")

res = collection_dpr.get(ids=["req_001"], include=["embeddings"])
print("stored embedding dim:", len(res["embeddings"][0]))

Successfully injected 100 requests into the vector database.
stored embedding dim: 768


In [9]:
collection_minilm = client.get_or_create_collection(name="collection_minilm")

# Inject all 100 items
collection_minilm.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids # embedding_dim=384 compared to 768 with DPR
)

print(f"Successfully injected {len(documents)} requests into the vector database.")

res = collection_minilm.get(ids=["req_001"], include=["embeddings"])
print("stored embedding dim:", len(res["embeddings"][0]))

Successfully injected 100 requests into the vector database.
stored embedding dim: 384


2 collections; one using DPR embedding, one using the default MiniLM embedding are created to evaluate which form of embedding yields a better result for retrieval

In [ ]:
# 1) Model set
dpr_qtoken = DPRQuestionEncoderTokenizer.from_pretrained('facebook/dpr-question_encoder-single-nq-base')
dpr_qenc = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
mini_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2) Create functions to convert sample queries into embeddings
def dpr_qemb(q): 
    return dpr_qenc(**dpr_qtoken(q, return_tensors="pt"))[0].detach().numpy()
def mini_qemb(q): 
    return mini_model.encode(q, normalize_embeddings=True)

# 3) retrieve function
def retrieve(collection, q_emb, k=10):
    return collection.query(query_embeddings=q_emb, n_results=k)["ids"][0]

# 4) Evaluation loop
def eval_retriever(collection, encoder, queries, gold, k=10):
    scores=[]
    for q, g in zip(queries, gold):
        preds = retrieve(collection, encoder(q), k)
        p10 = len(set(preds[:k]) & set(g)) / k
        scores.append(p10)
    return sum(scores)/len(scores)

# q_emb = dpr_qemb("example query")
# print(q_emb.shape) 
# print(collection_dpr.query(query_embeddings=q_emb.tolist(), n_results=5))
gold_ids = [[id] for id in ids]

print('DPR Hit Rate @ 10', eval_retriever(collection_dpr, dpr_qemb, documents, gold_ids, k=10))
print("MiniLM Hit Rate @ 10", eval_retriever(collection_minilm, mini_qemb, documents, gold_ids, k=10))


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24326.62it/s]
DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5260.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR P@10 0.09199999999999983
MiniLM P@10 0.09999999999999981


DPR ended up performing worse in this baseline test compared to MiniLM. 
P@10 measures Precision at 10, meaning how many of the top 10 search results returned by my retriever are actually relevant. In this case, a P@10 of 0.1 would be considered perfect, since each of the gold standard id lists only contain 1 id, meaning the maximum score is 1/10. Since MiniLM got a score of 0.09999999999999981, which is an almost 100% hit rate, it correctly found the most relevant result almost all the time. However, DPR only got a 0.092 P@10, meaning a 92% hit rate, which is comparatively worse than MiniLM. 

There are two reasons for this disparity:
1. Two Models vs One Model (Symmetric Evaluation)
In this current baseline test, my `queries` are the exact same text strings as my `documents`.
    - MiniLM: This is a single unified encoder (all-MiniLM-L6-v2). Since I pass the same exact text into the same exact model twice, it produces the same vector mathematics both times, so it's essentially searching for a string in a list by matching it to itself. Since the distance between the query and document is 0, it guarantees a 100% retrieval rate. The reason why it isn't exactly 0.1 is likely due to some rounding errors in the code itself.
    - DPR: This relies on a bi-encoder architecture. It uses two entirely different AI models:
        - A context encoder (dpr-ctx_encoder-single-nq-base)
        - A question encoder (dpr-question_encoder-single-nq-base)
Because the weights inside those two models are unique, passing the identical string into both models won't resultin identical vectors. so it is not looking for string equality. So it makes sense that it would have slightly lower P@10 score
2. NQ Dataset vs Ai Agency Requests (Domain Incompatibility)
The architecture of the DPR models I pulled from HuggingFace means it was strictly fine-tuned and trained on the NQ (Natural Questions) dataset.
    - NQ Data: This consists of real Google Server queries paired with Wikipedia articles. It heavily expects more trivia-style questions like "Who invented the lightbulb" mapping to factual paragraphs
    - My Synthetic Data: I have synthetic client requests which have totally different vocab and structure compared to the Wikipedia trivia.
MiniLM on the other hand is trained on over 1 billion diverse sentence pairs across Reddit, StackExchange, semantic similarity datasets, and more. It is way more generalised out-of-the-box for random text than DPR is. 

If I want to test DPR accurately, I will have to augment the documents from raw queries ("What kind of agent should I use for emails?") to an asymmetric setup ("Client Request: I need an automation agent to sort my inbound lead emails."). In this scenario, I think a generalised model like MiniLM might sometimes drop off while DPR might pull ahead. However, I won't be doing this since I feel like it's deliberately trying to give DPR a chance compared to objectively viewing it as DPR simply being unsuitable for this task. Perhaps later on I could do fine tuning for the DPR models to see what result they might have for this.

Actually on second thought it's because I am using the raw data from the original database, I am going to synthetically generate some 300 other distinct realistic questions and map them to document IDs instead.

In [ ]:
class GeneratedQueries(BaseModel):
    queries: list[str] = Field(description="A list of 3 distinct, realistic questions a client might ask that this document would answer.")

eval_data_path = "../data/eval_qa_dataset.json"
llm_client = get_llm_client()

if not os.path.exists(eval_data_path):
    print("Generating synthetic evaluation queries based on the documents. This will take ~7 minutes due to API rate limits...")
    eval_dataset = []
    
    for i, item in enumerate(data):
        doc_text = item['document']
        doc_id = item['id']
        prompt = f"Given this client request document: '{doc_text}'. Generate exactly 3 completely different short questions or ways a client might ask that this document would be the perfect answer for."
        
        # Keep retrying if we hit a quota limit
        while True:
            try:
                response = llm_client.models.generate_content(
                    model='gemini-3.1-flash-lite-preview',
                    contents=prompt,
                    config={
                        'response_mime_type': 'application/json',
                        'response_schema': GeneratedQueries,
                        'temperature': 0.7
                    }
                )
                generated = response.parsed.model_dump()["queries"]
                for q in generated:
                    eval_dataset.append({
                        "query": q,
                        "gold_ids": [doc_id]
                    })
                    
                # Print progress
                if (i+1) % 10 == 0:
                    print(f"[{i+1}/{len(data)}] Processed...")
                    
                break # Success, break out of retry loop
            except Exception as e:
                print(f"Rate limit hit! Waiting 10 seconds before retrying... ({e})")
                time.sleep(10)
                
        # Sleep for 4.5 seconds on every loop to respect the 15 Requests Per Minute free tier limit
        time.sleep(4.5)
            
    with open(eval_data_path, "w") as f:
        json.dump(eval_dataset, f, indent=4)
    print("Saved evaluation dataset to", eval_data_path)
else:
    with open(eval_data_path, "r") as f:
        eval_dataset = json.load(f)
    print("Loaded existing evaluation dataset with", len(eval_dataset), "queries.")

eval_queries = [item['query'] for item in eval_dataset]
eval_gold = [item['gold_ids'] for item in eval_dataset]


Generating synthetic evaluation queries based on the documents. This will take ~7 minutes due to API rate limits...
[10/100] Processed...
[20/100] Processed...
[30/100] Processed...
[40/100] Processed...
[50/100] Processed...
[60/100] Processed...
[70/100] Processed...
[80/100] Processed...
[90/100] Processed...
[100/100] Processed...
Saved evaluation dataset to ../data/eval_qa_dataset.json


In [38]:
print('DPR Hit Rate @ 10', eval_retriever(collection_dpr, dpr_qemb, eval_queries, eval_gold, k=10))
print("MiniLM Hit Rate @ 10", eval_retriever(collection_minilm, mini_qemb, eval_queries, eval_gold, k=10))

DPR Hit Rate @ 10 0.06033333333333329
MiniLM Hit Rate @ 10 0.09933333333333384


DPR still falls behind since as mentioned before, it has a domain mismatch. However, at least now I have a more accurate result using an actual evaluation dataset.

### Recursive Retrieval

Recursive Retrieval wouldn't be effective for the current system since it is usually only necessitated by larger documents, whereas I currently only have short 1-2 sentences in my sources.